In [0]:
%pip install "tabulate==0.9.0" "Unidecode==1.3.8" "darts[all]==0.45.0"


In [0]:
# Dependências do notebook são instaladas e fixadas na primeira célula.
# Não reinstalar pacotes intermediários: isso pode alterar a versão do Darts em execução.

In [0]:
dbutils.library.restartPython()

In [0]:
%pip install "tabulate==0.9.0" "Unidecode==1.3.8" -q

In [0]:
import pyspark.sql.functions as F
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import io
import pandas as pd
import matplotlib.pyplot as plt

In [0]:
import pandas

# Se o atributo não existir, criamos um alias para evitar o erro
if not hasattr(pandas.arrays, 'NumpyExtensionArray'):
    # Em versões recentes, a funcionalidade baseada em numpy mudou, 
    # mas muitas vezes isso é apenas uma verificação de tipo.
    # Podemos tentar apontar para a classe base de arrays de extensão.
    pandas.arrays.NumpyExtensionArray = pandas.api.extensions.ExtensionArray

#from darts.explainability.shap_explainer import ShapExplainer


In [0]:
from darts import TimeSeries, concatenate
from darts.utils.timeseries_generation import datetime_attribute_timeseries
from darts.models import (
    NaiveMean,
    NaiveSeasonal,
    ExponentialSmoothing,
    Prophet,
    LinearRegressionModel,
    LightGBMModel
)
from darts.metrics import mae, rmse, smape
from darts.timeseries import TimeSeries, concatenate
from darts.models import AutoARIMA, AutoCES, AutoMFLES, NaiveSeasonal, AutoETS, LinearRegressionModel, XGBModel, Prophet, Chronos2Model
from datetime import date, timedelta, datetime
import mlflow
mlflow.autolog(disable=True)

In [0]:
dbutils.widgets.text("catalog", "", "Catalog")
catalog = dbutils.widgets.get("catalog")
if not catalog:
    catalog = "planejamento_integrado"
print(catalog)

today_str = (datetime.utcnow() - timedelta(hours=3)).strftime("%Y-%m-%d")
#today_str = '2025-01-31'
today_date = datetime.strptime(today_str, "%Y-%m-%d").date()
print("today_str :", today_str, type(today_str))

yesterday_date = today_date - timedelta(days=1)
yesterday_str = yesterday_date.strftime("%Y-%m-%d")
print("yesterday_str:", yesterday_str)

In [0]:
ps = spark.sql(
    """
    select
    a.*
    from
    campinas_hvc_dev.gold.hc_atend_senha_medico_ps a
    WHERE hub LIKE 'Campinas' AND (dt_cancelamento is null) AND ds_SETOR_ATENDIMENTO LIKE '%HVC%'
    """
)

# Volume independente de PS

In [0]:
# === Observabilidade: setup ===
import os
import sys
import importlib

current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

import utils
importlib.reload(utils)
from utils import start_pipeline_log, end_pipeline_log

# timestamp usado como dt_referencia no log
timestamp = today_str.replace('-', '') + '_' + datetime.now().strftime('%H%M%S')


def get_databricks_context():
    job_id = None
    task_id = None
    notebook_path = None

    try:
        ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()

        job_id = ctx.jobId().get() if ctx.jobId().isDefined() else None
        task_id = ctx.currentRunId().get() if ctx.currentRunId().isDefined() else None
        notebook_path = ctx.notebookPath().get() if ctx.notebookPath().isDefined() else None

    except Exception as e:
        print(f"Could not get context: {e}")

    return job_id, task_id, notebook_path


job_id, task_id, notebook_path = get_databricks_context()


In [0]:
run_id = start_pipeline_log(
    spark=spark,
    pipeline_name="pi_previsao_cirurgias",
    step_name="previsao_urgentes.load_cirurgias_urgentes",
    job_id=job_id,
    task_id=task_id,
    notebook_path=notebook_path,
    dt_referencia=timestamp,
    source_table="campinas_hvc_dev.gold.cirurgia",
    target_table="cirurgias (não-eletivas, realizadas)"
)

try:
    cirurgia_source = spark.table('campinas_hvc_dev.gold.cirurgia')
    source_max_data_cirurgia = cirurgia_source.where(
        (F.col('hub') == 'Campinas') &
        (F.col('cd_estabelecimento') == 2) &
        (F.col('dt_inicio_real') < F.current_date())
    ).agg(F.max(F.to_date('dt_inicio_real')).alias('max_data')).first()['max_data']

    expected_source_max_date = yesterday_date
    if source_max_data_cirurgia != expected_source_max_date:
        raise RuntimeError(
            'Fonte desatualizada: campinas_hvc_dev.gold.cirurgia terminou em '
            f'{source_max_data_cirurgia}; esperado D-1 ({expected_source_max_date}).'
        )

    cirurgias = cirurgia_source\
        .select("hub", "cd_pessoa_fisica", "nr_cirurgia", "nr_atendimento", "dt_atualizacao", 'dt_inicio_real', 'dt_termino', "cd_estabelecimento", "dt_cancelamento", 'ds_procedimento', "ds_setor_atendimento", "ds_carater", "ds_status_cirurgia", "ds_setor_atend")\
        .where('hub = "Campinas"')\
        .where('cd_estabelecimento = 2')\
        .where(~F.col('ds_carater').like('%Eletiva%'))\
        .where(F.col('ds_status_cirurgia') == "Realizada")\
        .where(F.col('dt_inicio_real') < F.current_date())\
        .orderBy(F.desc('dt_inicio_real'))

    rows_written = cirurgias.count()
    end_pipeline_log(
        spark,
        run_id,
        status="SUCCESS",
        rows_written=rows_written,
    )

except Exception as e:
    safe_error = str(e).replace("'", "''")
    end_pipeline_log(
        spark,
        run_id,
        status="FAILED",
        error_message=safe_error
    )
    raise


In [0]:
run_id = start_pipeline_log(
    spark=spark,
    pipeline_name="pi_previsao_cirurgias",
    step_name="previsao_urgentes.aggregate_cirurgias_diarias",
    job_id=job_id,
    task_id=task_id,
    notebook_path=notebook_path,
    dt_referencia=timestamp,
    source_table="cirurgias",
    target_table="cirurgias_count"
)

try:
    cirurgias_count = cirurgias.groupBy(
        F.to_date('dt_inicio_real').alias("data_cirurgia")
    ).agg(
        F.count("*").alias("total_cirurgias_realizadas_dia")
    ).where(F.col('data_cirurgia').isNotNull())\
    .orderBy(F.col('data_cirurgia'))


    cirurgias_count.where(F.col('data_cirurgia') >= F.date_sub(F.current_date(), 365)).display()

    rows_written = cirurgias_count.count()
    end_pipeline_log(
        spark,
        run_id,
        status="SUCCESS",
        rows_written=rows_written,
    )

except Exception as e:
    safe_error = str(e).replace("'", "''")
    end_pipeline_log(
        spark,
        run_id,
        status="FAILED",
        error_message=safe_error
    )
    raise


In [0]:
cirurgias_pd = cirurgias_count.toPandas()

cirurgias_pd

In [0]:
# Configuração de estilo
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (15, 6)

# 1. Tratamento básico
cirurgias_pd['data_cirurgia'] = pd.to_datetime(cirurgias_pd['data_cirurgia'])
cirurgias_pd = cirurgias_pd.set_index('data_cirurgia').sort_index()

# A fonte ja foi validada em D-1; dias sem cirurgia urgente sao zero, nao
# uma lacuna que deve encurtar a serie usada pelo modelo.
complete_daily_index = pd.date_range(
    start=cirurgias_pd.index.min(), end=pd.Timestamp(yesterday_date), freq='D'
)
cirurgias_pd = cirurgias_pd.reindex(complete_daily_index, fill_value=0)
cirurgias_pd.index.name = 'data_cirurgia'

# 2. Plotagem da Série Temporal
plt.plot(cirurgias_pd.iloc[-365*2:], label='Cirurgias Urgentes', color='navy')
plt.title('Histórico Diário de Cirurgias Urgentes')
plt.ylabel('Qtd. Cirurgias')
plt.legend()
plt.show()

In [0]:
# Decomposição (Modelo Aditivo é comum, mas se a variação cresce com a média, use 'multiplicative')
resultado = seasonal_decompose(cirurgias_pd['total_cirurgias_realizadas_dia'].iloc[-365*2:], model='additive', period=7)

fig = resultado.plot()
plt.show()

In [0]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 10))
plot_acf(cirurgias_pd['total_cirurgias_realizadas_dia'], lags=40, ax=ax1) # 40 dias de lag
plot_pacf(cirurgias_pd['total_cirurgias_realizadas_dia'], lags=40, ax=ax2)
plt.show()

In [0]:
run_id = start_pipeline_log(
    spark=spark,
    pipeline_name="pi_previsao_cirurgias",
    step_name="previsao_urgentes.load_ps_completo",
    job_id=job_id,
    task_id=task_id,
    notebook_path=notebook_path,
    dt_referencia=timestamp,
    source_table="campinas_hvc_dev.gold.hc_atend_senha_medico_ps + hc_atendimento_paciente",
    target_table="ps (com agrupamento + internacao)"
)

try:
    ps = spark.sql(
        """
        select
        a.*,
        ( SELECT count(*) from bi_dev.gold.hc_atendimento_paciente aa where 1 = 1 and cast(aa.cd_pessoa_fisica as integer) = cast(a.cd_pessoa_fisica as integer) and aa.sigla_hub = a.sigla_hub and aa.ds_tipo_atendimento = "Internado" and to_timestamp(a.dt_alta_formatado, 'dd-MM-yyyy HH:mm') between aa.dt_entrada - interval '6' hour and aa.dt_entrada + interval '6' hour ) internacao
        from
        campinas_hvc_dev.gold.hc_atend_senha_medico_ps a
        WHERE hub LIKE 'Campinas' AND (dt_cancelamento is null) AND ds_setor_atendimento LIKE '%HVC%'
        """
    )

    de_para_agrupamento_novo = F.when(
            F.col('cid') == 'A90', 'Dengue'
        ).when(
            F.col('ds_clinica') == 'Pediátrica', 'Pediatria'
        ).when(
            (F.col('cid').isin(['R509', 'R05'])) | (F.col('cid_titulo') == 'Doenças do aparelho respiratório.'), 'Doenças do aparelho respiratório, Tosse e Febre'
        ).when(
            F.col('ds_clinica') == 'Ginecológica/Obstétrica', 'Ginecológica/Obstétrica'
        ).when(
            F.col('cid_titulo') == 'Algumas doenças infecciosas e parasitárias.', 'Doenças infecciosas e parasitárias'
        ).when(
            F.col('cid_titulo') == 'Doenças do aparelho geniturinário.', 'Doenças do aparelho geniturinário'
        ).when(
            (F.col('cid_titulo') == 'Doenças do aparelho circulatório.') | (F.col('cid').isin(['R074', 'R073', 'R002', 'R55', 'R000', 'R001', 'R071', 'R072', ])), 'Cardiologia'
        ).when(
            (F.col('ds_clinica') == 'Ortopédica'), 'Ortopedia'
        ).otherwise('Geral')

    # Regra de cardio: Doenças do aparelho circulatório ou
    # R074   DOR TORACICA, NAO ESPECIFICADA
    # R073   OUTRA DOR TORACICA
    # R002   PALPITACOES
    # R55   SINCOPE E COLAPSO
    # R000   TAQUICARDIA NAO ESPECIFICADA
    # R001   BRADICARDIA NAO ESPECIFICADA
    # R071   DOR TORACICA AO RESPIRAR
    # R072   DOR PRECORDIAL

    ps = ps.withColumn('agrupamento', de_para_agrupamento_novo)

    rows_written = ps.count()
    end_pipeline_log(
        spark,
        run_id,
        status="SUCCESS",
        rows_written=rows_written,
    )

except Exception as e:
    safe_error = str(e).replace("'", "''")
    end_pipeline_log(
        spark,
        run_id,
        status="FAILED",
        error_message=safe_error
    )
    raise


In [0]:
run_id = start_pipeline_log(
    spark=spark,
    pipeline_name="pi_previsao_cirurgias",
    step_name="previsao_urgentes.prep_ps_features",
    job_id=job_id,
    task_id=task_id,
    notebook_path=notebook_path,
    dt_referencia=timestamp,
    source_table="ps",
    target_table="ps_pd (pandas, pivot por agrupamento)"
)

try:
    ps_pd = ps\
        .withColumn('dt_referência', F.date_trunc('day', 'dt_entrada'))\
        .groupBy('dt_referência', 'agrupamento')\
        .agg(F.count('*').alias('qtd'), F.sum('internacao').alias('internacao'))\
        .fillna(0, subset='qtd')\
        .groupBy('dt_referência')\
        .pivot('agrupamento')\
        .sum('qtd')\
        .fillna(0)\
        .orderBy('dt_referência')\
        .toPandas()

    rows_written = len(ps_pd)
    end_pipeline_log(
        spark,
        run_id,
        status="SUCCESS",
        rows_written=rows_written,
    )

except Exception as e:
    safe_error = str(e).replace("'", "''")
    end_pipeline_log(
        spark,
        run_id,
        status="FAILED",
        error_message=safe_error
    )
    raise


In [0]:

# Criando dataframe ps_pd
ps_pd['dt_referência'] = pd.to_datetime(ps_pd['dt_referência'])
ps_pd = ps_pd.set_index('dt_referência').sort_index()

df_full = cirurgias_pd.join(ps_pd, how='left')

print(f"Dataset unificado: {df_full.shape[0]} linhas e {df_full.shape[1]} colunas.")
df_full.head(3)


# Previsão Teste

In [0]:
cirurgias_pd = cirurgias_pd.rename(columns={'total_cirurgias_realizadas_dia': 'cirurgias'}).fillna(0)

In [0]:
ts = TimeSeries.from_dataframe(cirurgias_pd, freq='D', fillna_value=0)
ps = TimeSeries.from_dataframe(ps_pd, freq='D', fillna_value=0)

In [0]:
ts.to_dataframe().isna().sum()

In [0]:
def bias_direcional(actual_series, pred_series):
    cut_actual = actual_series.slice_intersect(pred_series)
    cut_pred = pred_series.slice_intersect(cut_actual)

    diff = cut_pred.values() - cut_actual.values()
    acima = (diff > 0).sum()
    abaixo = (diff < 0).sum()
    total = len(diff)
    return ((acima - abaixo) / total) * 100

def bias_pct(actual_series, pred_series):
    cut_actual = actual_series.slice_intersect(pred_series)
    cut_pred = pred_series.slice_intersect(cut_actual)

    erro = cut_pred.values() - cut_actual.values()

    return (erro.sum() / cut_actual.values().sum()) * 100

In [0]:
run_id = start_pipeline_log(
    spark=spark,
    pipeline_name="pi_previsao_cirurgias",
    step_name="previsao_urgentes.build_covariates",
    job_id=job_id,
    task_id=task_id,
    notebook_path=notebook_path,
    dt_referencia=timestamp,
    source_table="ps (TimeSeries) + {catalog}.gold.calendario",
    target_table="covariates_feriados"
)

try:
    cov_weekly = datetime_attribute_timeseries(
        ps.time_index, 
        attribute="dayofweek", 
        cyclic=True
    )

    cov_annual = datetime_attribute_timeseries(
        ps.time_index, 
        attribute="month", 
        cyclic=True
    )

    encoders = cov_weekly.stack(cov_annual)

    encoders_common = encoders.slice_intersect(ps)
    ps_common = ps.slice_intersect(encoders)

    full_covariates = encoders_common.stack(ps_common)

    full_covariates_pd = full_covariates.to_dataframe()

    feriados = spark.table(f'{catalog}.gold.calendario')\
        .where("cidade = 'Campinas'")\
        .withColumn("feriado", F.when(F.col("feriado").isNotNull(), 1).otherwise(0))\
        .withColumn(
            "feriado",
            F.when(
                (F.dayofweek("data").between(2, 6)),  # 2=Monday, 6=Friday
                F.col("feriado")
            ).otherwise(F.lit(0))
        )\
        .select(
            F.col('data').alias('dt_referência'),
            F.col('feriado'),
            F.when(F.col('ferias_escolares'), 1).otherwise(0).alias('ferias_escolares')
        )\
        .withColumn('sabado', F.when(F.dayofweek('dt_referência') == 7, 1).otherwise(0))\
        .withColumn('domingo', F.when(F.dayofweek('dt_referência') == 1, 1).otherwise(0))\
        .orderBy('dt_referência').toPandas().set_index('dt_referência')

    covariates_feriados = full_covariates_pd.merge(
        feriados,  
        left_index=True,  
        right_index=True
    )

    covariates_feriados = TimeSeries.from_dataframe(
        covariates_feriados,
        freq='D'
    )

    rows_written = len(covariates_feriados)
    end_pipeline_log(
        spark,
        run_id,
        status="SUCCESS",
        rows_written=rows_written,
    )

except Exception as e:
    safe_error = str(e).replace("'", "''")
    end_pipeline_log(
        spark,
        run_id,
        status="FAILED",
        error_message=safe_error
    )
    raise


# Final

In [0]:
date_90_days_ago = date.today() - timedelta(days=90)
date_90_days_ago

In [0]:
ps_columns = list(ps.columns)
ps_columns

In [0]:
models = [
    ('Naive Seasonal', NaiveSeasonal(K=7)), 
    ('Exp. Smoothing', ExponentialSmoothing()),
    ('AutoARIMA', AutoARIMA()),
    ('AutoETS', AutoETS()),
    ('Chronos90', Chronos2Model(input_chunk_length=90, output_chunk_length=30)),
    ('Linear Regression_30', LinearRegressionModel(lags=14, output_chunk_length=30)),
    ('Prophet', Prophet(country_holidays='BR')),
    ('LGBM', LightGBMModel(lags=14, lags_future_covariates=[0])),
    ('LGBM_2', LightGBMModel(lags=14, lags_future_covariates=[-7, -6, -5, -4, -3, -2, -1, 0])),
]

In [0]:
run_id = start_pipeline_log(
    spark=spark,
    pipeline_name="pi_previsao_cirurgias",
    step_name="previsao_urgentes.backtest_models",
    job_id=job_id,
    task_id=task_id,
    notebook_path=notebook_path,
    dt_referencia=timestamp,
    source_table="ts + covariates_feriados",
    target_table="forecasts (dict por modelo)"
)

try:
    # Falhe antes do backtest caro quando a previsao de PS da origem ainda
    # nao estiver pronta para os 365 dias que o predict operacional exige.
    ps_preflight = spark.table(f'{catalog}.ds_pipeline.previsao_demanda_ps').where(
        F.col('data_processamento') == F.current_date()
    ).agg(
        F.count('*').alias('row_count'),
        F.countDistinct(F.to_date('dt_referencia')).alias('distinct_dates'),
        F.min(F.to_date('dt_referencia')).alias('min_date'),
        F.max(F.to_date('dt_referencia')).alias('max_date')
    ).first()
    expected_ps_start = today_date
    expected_ps_end = today_date + timedelta(days=364)
    if (
        ps_preflight['row_count'] != 365
        or ps_preflight['distinct_dates'] != 365
        or ps_preflight['min_date'] != expected_ps_start
        or ps_preflight['max_date'] != expected_ps_end
    ):
        raise RuntimeError(
            'Fonte desatualizada: '
            f'{catalog}.ds_pipeline.previsao_demanda_ps possui '
            f"{ps_preflight['row_count']} linhas, de {ps_preflight['min_date']} "
            f"a {ps_preflight['max_date']}; esperado 365 linhas de "
            f'{expected_ps_start} a {expected_ps_end}.'
        )

    forecast_horizon = 30
    stride = 30
    start_date = pd.to_datetime(date_90_days_ago)

    results = []
    forecasts = {} 

    print(f"Iniciando Backtesting com Encoders (Sazonalidade Anual + Semanal + Feriados + Ferias)...\n")

    for model_name, model in models:
        if model.supports_future_covariates:
            covariates_to_use = covariates_feriados[ps_columns + ['ferias_escolares']]
        else:
            covariates_to_use = None

        try:
            hf = model.historical_forecasts(
                series=ts,
                future_covariates=covariates_to_use,
                start=start_date,
                forecast_horizon=forecast_horizon,
                stride=stride,
                last_points_only=False,
                retrain=True,
                verbose=False,
            )

            hf_total = concatenate(hf)
            forecasts[model_name] = hf_total

            # Métricas
            res_mae = mae(ts, hf_total)
            res_rmse = rmse(ts, hf_total)
            res_smape = smape(ts, hf_total)
            res_bias_direcional = bias_direcional(ts, hf_total)
            res_bias_pct = bias_pct(ts, hf_total)
            # res_mase = mase(ts, hf_total, m=7)

            results.append({
                'Modelo': model_name,
                'MAE': res_mae,
                'RMSE': res_rmse,
                'SMAPE (%)': res_smape,
                'Bias Direcional (%)': res_bias_direcional,
                'Bias (%)': res_bias_pct
                # 'MASE': res_mase
            })
            print(f"✅ {model_name} processado.")

        except Exception as e:
            print(f"❌ Erro em {model_name}: {e}")

    df_results = pd.DataFrame(results).sort_values(by='MAE')
    print("\n--- Resultados com Encoders ---")
    print(df_results.to_markdown(index=False, floatfmt=".2f"))

    plt.figure(figsize=(15, 7))
    plot_start = start_date - pd.Timedelta(days=60)

    ts.drop_before(plot_start).plot(label='REAL', color='black', linewidth=1)

    for model_name, forecast_series in forecasts.items():
        ls = '--' if 'Linear' in model_name else '-'
        if model_name in ['Chronos90', 'Prophet', 'LGBM_2']:
            forecast_series.plot(label=model_name, linewidth=1.5, linestyle=ls, alpha=0.8)

    plt.title('Comparação de Modelos (Incluindo Sazonalidade Anual/Semanal)', fontsize=14)
    plt.legend(loc='best')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    rows_written = len(forecasts)
    end_pipeline_log(
        spark,
        run_id,
        status="SUCCESS",
        rows_written=rows_written,
    )

except Exception as e:
    safe_error = str(e).replace("'", "''")
    end_pipeline_log(
        spark,
        run_id,
        status="FAILED",
        error_message=safe_error
    )
    raise


In [0]:
ts.slice_intersect(forecasts['Prophet']).plot(label='Cirurgias Urgência e Emergência')
forecasts['Prophet'].plot(label='Previsão')

In [0]:
weekly_real = ts.slice_intersect(forecasts['Prophet']).resample(freq='W', method='sum')[1:-1].to_dataframe()
weekly_chronos = forecasts['Chronos90'].resample(freq='W', method='sum')[1:-1].to_dataframe()
weekly_prophet = forecasts['Prophet'].resample(freq='W', method='sum')[1:-1].to_dataframe()

df_plot = pd.DataFrame({
    'Real': weekly_real.squeeze(),
    'Chronos90': weekly_chronos.squeeze(),
    'Predição': weekly_prophet.squeeze()
}, index=weekly_real.index)

ax = df_plot.plot(kind='bar', figsize=(15, 6))
ax.set_xticklabels([d.strftime('%Y-%m-%d') for d in df_plot.index], rotation=45)
plt.title('Comparação semanal')
plt.ylabel('Soma semanal')
plt.xlabel('Semana')
plt.tight_layout()
plt.show()

In [0]:
weekly_real = ts.slice_intersect(forecasts['Prophet']).resample(freq='W', method='sum')[1:-1]
weekly_prophet = forecasts['Prophet'].resample(freq='W', method='sum')[1:-1]

print('Métricas Semanais')

print(f"MAE: {mae(weekly_real, weekly_prophet):.2f}")
print(f"RMSE: {rmse(weekly_real, weekly_prophet):.2f}")
print(f"SMAPE (%): {smape(weekly_real, weekly_prophet):.2f}")
print(f"Bias Direcional: {bias_direcional(weekly_real, weekly_prophet):.2f}")
print(f"Bias (%): {bias_pct(weekly_real, weekly_prophet):.2f}")


# Previsão Futura

In [0]:
run_id = start_pipeline_log(
    spark=spark,
    pipeline_name="pi_previsao_cirurgias",
    step_name="previsao_urgentes.load_ps_futuro",
    job_id=job_id,
    task_id=task_id,
    notebook_path=notebook_path,
    dt_referencia=timestamp,
    source_table="{catalog}.ds_pipeline.previsao_demanda_ps + {catalog}.gold.calendario",
    target_table="future_covariates_predict"
)

try:
    ps_futuro = spark.table(f'{catalog}.ds_pipeline.previsao_demanda_ps')\
        .where(F.col('data_processamento') == F.current_date())\
        .orderBy('dt_referencia')

    feriados = spark.table(f'{catalog}.gold.calendario')\
        .where("cidade = 'Campinas'")\
        .select(
            F.col('data').alias('dt_referencia'),
            F.when(F.col('ferias_escolares'), 1).otherwise(0).alias('ferias_escolares')
        )

    mapeamento_agrupamento = {
        'cardiologia': 'Cardiologia',
        'dengue': 'Dengue',
        'doencas_do_aparelho_geniturinario': 'Doenças do aparelho geniturinário',
        'doencas_do_aparelho_respiratorio_tosse_e_febre': 'Doenças do aparelho respiratório, Tosse e Febre',
        'doencas_infecciosas_e_parasitarias': 'Doenças infecciosas e parasitárias',
        'geral': 'Geral',
        'ginecologicaobstetrica': 'Ginecológica/Obstétrica',
        'ortopedia': 'Ortopedia',
        'pediatria': 'Pediatria'
    }

    ps_futuro = ps_futuro\
        .withColumnsRenamed(mapeamento_agrupamento)\
        .join(feriados, on='dt_referencia', how='left')\
        .drop('versao', 'distancia_previsao', 'data_processamento')

    # Convert to pandas and handle duplicates
    ps_futuro_pd = ps_futuro.toPandas()

    # Check for duplicates
    duplicates_mask = ps_futuro_pd.duplicated(subset='dt_referencia', keep=False)
    num_duplicate_rows = duplicates_mask.sum()

    if num_duplicate_rows > 0:
        print(f"⚠️  Found {num_duplicate_rows} duplicate rows for dt_referencia")
        print(f"   Number of unique dates affected: {ps_futuro_pd[duplicates_mask]['dt_referencia'].nunique()}")
        print("\nSample of duplicate rows (sorted by dt_referencia):")
        print(ps_futuro_pd[duplicates_mask].sort_values('dt_referencia').head(10))
        print("\n")
    else:
        print("✅ No duplicate rows found")

    # Group by dt_referencia and aggregate (sum numeric columns, take first for others)
    ps_futuro_pd = ps_futuro_pd.groupby('dt_referencia').agg({
        col: 'sum' if ps_futuro_pd[col].dtype in ['int64', 'float64'] else 'first'
        for col in ps_futuro_pd.columns if col != 'dt_referencia'
    }).reset_index()

    ps_futuro_pd = ps_futuro_pd.set_index('dt_referencia')

    future_covariates_predict = TimeSeries.from_dataframe(
        ps_futuro_pd,
        freq='D')

    rows_written = len(future_covariates_predict)
    end_pipeline_log(
        spark,
        run_id,
        status="SUCCESS",
        rows_written=rows_written,
    )

except Exception as e:
    safe_error = str(e).replace("'", "''")
    end_pipeline_log(
        spark,
        run_id,
        status="FAILED",
        error_message=safe_error
    )
    raise


In [0]:
ps_futuro.columns

In [0]:
run_id = start_pipeline_log(
    spark=spark,
    pipeline_name="pi_previsao_cirurgias",
    step_name="previsao_urgentes.train_predict_prophet",
    job_id=job_id,
    task_id=task_id,
    notebook_path=notebook_path,
    dt_referencia=timestamp,
    source_table="ts + future_covariates_predict",
    target_table="predictions (Prophet 365d)"
)

try:
    model = Prophet(country_holidays='BR')

    forecast_horizon = 365

    expected_target_end = pd.Timestamp(yesterday_date)
    if ts.end_time().normalize() != expected_target_end:
        raise RuntimeError(
            'Fonte desatualizada: campinas_hvc_dev.gold.cirurgia gerou serie '
            f'até {ts.end_time().date()}; esperado D-1 ({yesterday_date}).'
        )

    expected_covariate_index = pd.date_range(
        start=pd.Timestamp(today_date), periods=forecast_horizon, freq='D'
    )
    actual_covariate_index = pd.DatetimeIndex(
        future_covariates_predict.time_index
    ).normalize()
    if not actual_covariate_index.equals(expected_covariate_index):
        raise RuntimeError(
            'Fonte desatualizada: '
            f'{catalog}.ds_pipeline.previsao_demanda_ps nao contem os 365 dias '
            f'de {today_date} a {today_date + timedelta(days=364)}.'
        )

    print("\n=== DEBUG (ANTES DO PREDICT) ===")

    last_target = ts.end_time()
    last_cov = future_covariates_predict.end_time()

    print("Última data target:", last_target)
    print("Última data covariates:", last_cov)
    print("Tamanho covariates:", len(future_covariates_predict))

    # Data mínima necessária
    required_end = last_target + forecast_horizon * ts.freq
    print("Data mínima esperada:", required_end)

    # Checagem antecipada
    if last_cov < required_end:
        print("FALTAM DATAS NAS COVARIATES")

        expected_future_dates = set(
            last_target + i * ts.freq for i in range(1, forecast_horizon + 1)
        )

        cov_dates = set(future_covariates_predict.time_index)

        missing_dates = sorted(expected_future_dates - cov_dates)

        print(f"Quantidade de datas faltantes: {len(missing_dates)}")

        print("Exemplo de datas faltantes:")
        print(missing_dates[:10])

    # Execução do modelo (sem duplicar debug)
    try:
        model.fit(
            series=ts,
            future_covariates=covariates_feriados[ps_columns + ['ferias_escolares']],
        )

        predictions = model.predict(
            n=forecast_horizon,
            future_covariates=future_covariates_predict
        )

    except Exception as e:
        print("\n ERRO NA PREVISÃO:", e)
        raise e

    rows_written = len(predictions)
    end_pipeline_log(
        spark,
        run_id,
        status="SUCCESS",
        rows_written=rows_written,
    )

except Exception as e:
    safe_error = str(e).replace("'", "''")
    end_pipeline_log(
        spark,
        run_id,
        status="FAILED",
        error_message=safe_error
    )
    raise


In [0]:
df_pred = predictions.to_dataframe().round(0)
df_pred2 = predictions.to_dataframe()
df_pred.plot(figsize=(15,6), label='Arredondado')
df_pred2.plot(ax=plt.gca(), label='Original')
plt.legend()
plt.show()


# Pós-processamento

In [0]:
run_id = start_pipeline_log(
    spark=spark,
    pipeline_name="pi_previsao_cirurgias",
    step_name="previsao_urgentes.post_processing",
    job_id=job_id,
    task_id=task_id,
    notebook_path=notebook_path,
    dt_referencia=timestamp,
    source_table="predictions",
    target_table="prev_final (pandas com previsao_emergencia_e_urgencia + dt_referencia)"
)

try:
    prev_final = predictions.to_dataframe().reset_index()\
      .rename(columns={'cirurgias': 'previsao_emergencia_e_urgencia',
                       'data_cirurgia': 'dt_referencia'})

    # Arredondando
    prev_final['previsao_emergencia_e_urgencia'] = prev_final['previsao_emergencia_e_urgencia'].round(0).astype(int)

    # Limitando em 0
    prev_final['previsao_emergencia_e_urgencia'] = prev_final['previsao_emergencia_e_urgencia'].clip(lower=0)

    # Adicionando data_processamento
    prev_final['data_processamento'] = today_date

    # Adicionando distancia_previsao como número inteiro de dias
    prev_final['distancia_previsao'] = (prev_final['dt_referencia'].dt.date - today_date).apply(lambda x: x.days + 1).astype(int)

    prev_final

    rows_written = len(prev_final)
    end_pipeline_log(
        spark,
        run_id,
        status="SUCCESS",
        rows_written=rows_written,
    )

except Exception as e:
    safe_error = str(e).replace("'", "''")
    end_pipeline_log(
        spark,
        run_id,
        status="FAILED",
        error_message=safe_error
    )
    raise


In [0]:
resultado_spark = spark.createDataFrame(prev_final).withColumn('import_date', F.current_timestamp())
resultado_spark.display()

# Salvando

In [0]:
from pathlib import Path
import sys
import importlib
import os
import unidecode

current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
sys.path.append(parent_dir)

import utils
importlib.reload(utils)
from utils import write_to_table

In [0]:
run_id = start_pipeline_log(
    spark=spark,
    pipeline_name="pi_previsao_cirurgias",
    step_name="previsao_urgentes.save_previsao_urgencia_emergencia",
    job_id=job_id,
    task_id=task_id,
    notebook_path=notebook_path,
    dt_referencia=timestamp,
    source_table="resultado_spark",
    target_table="{catalog}.ds_pipeline.previsao_urgencia_emergencia"
)

try:
    table_name = f"{catalog}.ds_pipeline.previsao_urgencia_emergencia"

    write_to_table(
        df=resultado_spark,
        table_name=table_name,
        data_column="data_processamento",
        today_date=today_date
    )

    # Display recent data
    display(
        spark.table(table_name)
             .where(F.col("data_processamento") == F.to_date(F.lit(today_date)))
             .orderBy(F.desc("data_processamento"), "dt_referencia")
             .limit(1000)
    )

    rows_read = resultado_spark.count()
    rows_written = resultado_spark.count()
    end_pipeline_log(
        spark,
        run_id,
        status="SUCCESS",
        rows_read=rows_read,
        rows_written=rows_written,
    )

except Exception as e:
    safe_error = str(e).replace("'", "''")
    end_pipeline_log(
        spark,
        run_id,
        status="FAILED",
        error_message=safe_error
    )
    raise


In [0]:
spark.table(table_name).limit(1000).display()